# APEX / RESEARCH — Experiment 001
## Formula One lap-time prediction

**Core question:** Can `tire_age` explain the next lap?

This notebook is the authoritative analytical artifact for the experiment. It loads the supplied CSV archive, verifies race eligibility, removes pit-transition/outlier laps, reconstructs stints and `tire_age`, enforces an entire-final-stint holdout, trains two regressors, evaluates RMSE/MAE, draws the final-stint chart, and writes the same result artifacts consumed by the website.

The supplied archive does not contain weather, track-condition, tire-compound, or red-flag columns. The selected 2011 Australian Grand Prix is therefore accepted only because its race-day condition and no-red-flag status are recorded as external provenance in `experiment/run_experiment.py` and `results/summary.json`; those labels are not inferred from lap times.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from experiment.run_experiment import (
    COMPLETED_STATUS,
    OUTLIER_MULTIPLIER,
    RACE_PROVENANCE,
    RANDOM_SEED,
    clean_laps_for_race,
    find_race_selection,
    read_csv,
    save_final_stint_plot,
    train_and_evaluate,
    write_artifacts,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Random seed: {RANDOM_SEED}")

Project root: /home/user
Random seed: 42


## 1. Load the supplied input tables

In [2]:
input_names = [
    "races.csv",
    "results.csv",
    "lap_times.csv",
    "pit_stops.csv",
    "status.csv",
    "drivers.csv",
]
frames = {name: read_csv(name) for name in input_names}
races = frames["races.csv"]
results = frames["results.csv"]
lap_times = frames["lap_times.csv"]
pit_stops = frames["pit_stops.csv"]
statuses = frames["status.csv"]
drivers = frames["drivers.csv"]

source_shapes = {name: list(frame.shape) for name, frame in frames.items()}
pd.DataFrame(source_shapes, index=["rows", "columns"]).T

,rows,columns
races.csv,1125,18
results.csv,26759,18
lap_times.csv,589081,6
pit_stops.csv,11371,7
status.csv,139,2
drivers.csv,861,9


## 2. Inspect and validate schemas

In [3]:
required_schema = {
    "races.csv": {"raceId", "year", "round", "name", "date", "sprint_date"},
    "results.csv": {"raceId", "driverId", "grid", "statusId", "laps"},
    "lap_times.csv": {"raceId", "driverId", "lap", "milliseconds"},
    "pit_stops.csv": {"raceId", "driverId", "stop", "lap", "milliseconds"},
}
for filename, columns in required_schema.items():
    missing = columns.difference(frames[filename].columns)
    assert not missing, f"{filename} missing required columns: {sorted(missing)}"

assert races["raceId"].notna().all()
assert results[["raceId", "driverId", "grid", "statusId"]].notna().all().all()
assert lap_times[["raceId", "driverId", "lap", "milliseconds"]].notna().all().all()
assert pit_stops[["raceId", "driverId", "lap", "milliseconds"]].notna().all().all()
print("Required schemas and key-field null checks passed.")

Required schemas and key-field null checks passed.


## 3. Audit and verify the selected race

In [4]:
selection, eligible_races = find_race_selection(
    races, results, lap_times, pit_stops, statuses
)
selected_audit = eligible_races[eligible_races["selected"]].copy()
selected_audit.T

,839
raceId,841
year,2011
round,1
name,Australian Grand Prix
date,2011-03-27
sprint_date,NaN
completed_drivers,7
lap_drivers,22
pit_drivers,21
pit_events,45


In [5]:
assert selection.dry_verified
assert selection.no_red_flag_verified
assert 5 <= selection.completed_drivers <= 10
assert not selection.sprint_scheduled
assert selection.race_id in RACE_PROVENANCE

verification = {
    "race": f"{selection.year} {selection.name}",
    "race_id": selection.race_id,
    "dry_verified": selection.dry_verified,
    "dry_sources": list(selection.dry_sources),
    "no_red_flag_verified": selection.no_red_flag_verified,
    "no_red_flag_sources": list(selection.no_red_flag_sources),
    "completed_drivers": selection.completed_drivers,
    "lap_drivers": selection.lap_drivers,
    "pit_drivers": selection.pit_drivers,
    "pit_events": selection.pit_events,
    "sprint_scheduled": selection.sprint_scheduled,
}
print(json.dumps(verification, indent=2))

{
  "race": "2011 Australian Grand Prix",
  "race_id": 841,
  "dry_verified": true,
  "dry_sources": [
    "https://www.racefans.net/2011/03/24/australian-grand-prix-race-weekend-programme-2/",
    "https://www.autosport.com/f1/live-text/2011-australian-grand-prix-australian-grand-prix-weather-46090/46090/"
  ],
  "no_red_flag_verified": true,
  "no_red_flag_sources": [
    "https://racingnews365.com/does-formula-1-use-the-red-flag-too-often-nowadays"
  ],
  "completed_drivers": 7,
  "lap_drivers": 22,
  "pit_drivers": 21,
  "pit_events": 45,
  "sprint_scheduled": false
}


## 4. Clean lap observations and reconstruct stints

In [6]:
cleaned, cleaning, completed = clean_laps_for_race(
    selection, results, statuses, lap_times, pit_stops, drivers
)
cleaning_summary = pd.DataFrame(cleaning["summary_rows"])
display(cleaning_summary)
display(cleaning["detail"])

summary_values = dict(zip(cleaning_summary["metric"], cleaning_summary["value"]))
assert summary_values["raw_laps"] == summary_values["removed_laps"] + summary_values["retained_laps"]
assert summary_values["removed_laps"] == (
    summary_values["pit_stop_laps"]
    + summary_values["post_pit_laps"]
    + summary_values["extreme_slow_laps"]
)

,metric,value
0,raw_laps,406.0
1,pit_stop_laps,18.0
2,post_pit_laps,18.0
3,extreme_slow_laps,0.0
4,removed_laps,36.0
5,retained_laps,370.0
6,completed_drivers,7.0
7,driver_median_multiplier,1.5
8,eligible_race_id,841.0


,driver_id,driver,driver_name,raw_laps,pit_stop_laps,post_pit_laps,extreme_slow_laps,removed_laps,retained_laps,driver_median_ms,stints,final_stint,final_stint_laps
0,1,HAM,Lewis Hamilton,58,2,2,0,4,54,91777.5,3,3,21
1,4,ALO,Fernando Alonso,58,3,3,0,6,52,91107.5,4,4,15
2,13,MAS,Felipe Massa,58,3,3,0,6,52,92453.0,4,4,9
3,17,WEB,Mark Webber,58,3,3,0,6,52,91367.0,4,4,16
4,18,BUT,Jenson Button,58,3,3,0,6,52,91326.5,3,4,20
5,20,VET,Sebastian Vettel,58,2,2,0,4,54,91308.0,3,3,21
6,808,PET,Vitaly Petrov,58,2,2,0,4,54,91785.5,3,3,21


In [7]:
# Exact feature convention: initial stint is anchored at lap 0; after a pit,
# tire_age = current lap - latest pit lap. Pit and immediate post-pit rows
# are removed before the feature is used by a model.
preview_columns = [
    "driver", "lap", "last_pit_lap", "tire_age", "stint", "split",
    "milliseconds", "lap_time_seconds"
]
display(cleaned[preview_columns].head(16))
assert (cleaned["tire_age"] == cleaned["lap"] - cleaned["last_pit_lap"]).all()
assert (cleaned["stint"] >= 1).all()
assert (cleaning["detail"]["stints"] >= 2).all()
print("Every completed driver has enough stints for a final-stint holdout.")

,driver,lap,last_pit_lap,tire_age,stint,split,milliseconds,lap_time_seconds
0,HAM,1,0,1,1,train,100573,100.573
1,HAM,2,0,2,1,train,93774,93.774
2,HAM,3,0,3,1,train,92900,92.900
3,HAM,4,0,4,1,train,92582,92.582
4,HAM,5,0,5,1,train,92471,92.471
5,HAM,6,0,6,1,train,92434,92.434
6,HAM,7,0,7,1,train,92447,92.447
7,HAM,8,0,8,1,train,92310,92.310
8,HAM,9,0,9,1,train,92612,92.612
9,HAM,10,0,10,1,train,93121,93.121


Every completed driver has enough stints for a final-stint holdout.


## 5. Leakage-safe split verification

In [8]:
train = cleaned[cleaned["split"].eq("train")].copy()
test = cleaned[cleaned["split"].eq("test")].copy()
assert not train.empty and not test.empty

for driver_id, driver_rows in cleaned.groupby("driver_id"):
    final_stint = driver_rows["stint"].max()
    driver_train = train[train["driver_id"].eq(driver_id)]
    driver_test = test[test["driver_id"].eq(driver_id)]
    assert (driver_test["stint"] == final_stint).all()
    assert (driver_train["stint"] < final_stint).all()

train_keys = set(zip(train["driver_id"], train["lap"]))
test_keys = set(zip(test["driver_id"], test["lap"]))
assert train_keys.isdisjoint(test_keys)
print(f"Earlier-stint training rows: {len(train)}")
print(f"Entire-final-stint test rows: {len(test)}")
print("No final-stint rows appear in training and no train/test lap key overlaps exist.")

Earlier-stint training rows: 247
Entire-final-stint test rows: 123
No final-stint rows appear in training and no train/test lap key overlaps exist.


## 6. Train and evaluate both model families

In [9]:
comparison, prediction_rows, model_summary = train_and_evaluate(cleaned)
comparison[["model", "feature_set", "features", "rmse", "mae", "train_rows", "test_rows"]]

,model,feature_set,features,rmse,mae,train_rows,test_rows
0,Random Forest,baseline,grid + lap,1.267923,1.166832,247,123
1,Random Forest,enhanced,grid + lap + tire_age,0.944517,0.753847,247,123
2,Gradient Boosting,baseline,grid + lap,1.063629,0.917512,247,123
3,Gradient Boosting,enhanced,grid + lap + tire_age,0.948892,0.755878,247,123


## 7. Final-stint actual versus predicted visualization

In [10]:
best = model_summary["best"]
final_rows = prediction_rows[prediction_rows["split"].eq("test")].copy()
longest_final = (
    final_rows.groupby(["driver_id", "driver", "driver_name"], as_index=False)
    .size()
    .sort_values(["size", "driver_id"], ascending=[False, True])
    .iloc[0]
)
selected_driver_rows = final_rows[final_rows["driver_id"].eq(int(longest_final["driver_id"]))].sort_values("lap")

fig, ax = plt.subplots(figsize=(11, 4.8), dpi=140)
ax.plot(selected_driver_rows["lap"], selected_driver_rows["lap_time_seconds"], "o-", color="#e9493f", label="Actual")
ax.plot(selected_driver_rows["lap"], selected_driver_rows["predicted"], "--", color="#5b8588", label="Predicted")
ax.set_title(
    f"Final stint — {longest_final['driver_name']} / {best['model']} / {best['feature_set']}"
)
ax.set_xlabel("Lap number")
ax.set_ylabel("Lap time (seconds)")
ax.grid(alpha=0.18)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print(f"Selected final-stint driver: {longest_final['driver_name']}")
print(f"Held-out rows: {len(selected_driver_rows)}")

Selected final-stint driver: Lewis Hamilton
Held-out rows: 21


## 8. Save all artifacts from this same run

In [11]:
write_artifacts(
    selection=selection,
    audit=eligible_races,
    cleaned=cleaned,
    cleaning=cleaning,
    completed=completed,
    comparison=comparison,
    prediction_rows=prediction_rows,
    model_summary=model_summary,
    source_shapes=source_shapes,
)

artifact_paths = [
    "results/cleaning_summary.csv",
    "results/cleaning_by_driver.csv",
    "results/eligible_races.csv",
    "results/model_comparison.csv",
    "results/stint_predictions.csv",
    "results/summary.json",
    "results/final_stint_actual_vs_predicted.png",
]
for artifact in artifact_paths:
    path = PROJECT_ROOT / artifact
    assert path.exists(), path
    print(path)

/home/user/results/cleaning_summary.csv
/home/user/results/cleaning_by_driver.csv
/home/user/results/eligible_races.csv
/home/user/results/model_comparison.csv
/home/user/results/stint_predictions.csv
/home/user/results/summary.json
/home/user/results/final_stint_actual_vs_predicted.png


## 9. Findings, stated carefully

In [12]:
baseline_rmse = comparison.loc[comparison["feature_set"].eq("baseline"), "rmse"].mean()
enhanced_rmse = comparison.loc[comparison["feature_set"].eq("enhanced"), "rmse"].mean()
difference = enhanced_rmse - baseline_rmse
change_pct = difference / baseline_rmse * 100

print(f"Observed mean baseline RMSE across model families: {baseline_rmse:.6f} s")
print(f"Observed mean enhanced RMSE across model families: {enhanced_rmse:.6f} s")
print(f"Observed difference: {difference:+.6f} s ({change_pct:+.2f}%)")
print(
    "Interpretation: this is an observed predictive comparison on the held-out "
    "final stints. It does not establish that tire age causes slower lap times."
)

Observed mean baseline RMSE across model families: 1.165776 s
Observed mean enhanced RMSE across model families: 0.946705 s
Observed difference: -0.219072 s (-18.79%)
Interpretation: this is an observed predictive comparison on the held-out final stints. It does not establish that tire age causes slower lap times.


### Reproducibility note

Run this notebook from the repository root with the packages pinned in `requirements.txt`. The notebook calls the same experiment functions used by `python experiment/run_experiment.py`, validates the split before fitting, and writes the artifacts consumed by the website. No random train/test split is used.